In [3]:
# =========================================================
# SECTION 1: SETUP
# Cross-dataset stability notebook:
# - shared variables only
# - LAID kept, LAI removed
# - compare baseline vs TS-engineered LightGBM
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
import pandas as pd
import lightgbm as lgb

from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Paths
PRIMARY_DATA_PATH = Path("DSSAT_Merged_Neat.csv")
SECONDARY_DATA_PATH = Path("DSSAT_Final_weather_only_until_2050259.csv")

# Core config
TARGET_COL = "y_enter5_7"
GROUP_COL = "RUNNO"

# We will order by DAP for all within-run time operations
ORDER_COL = "DAP"

# TS feature settings (used later for Model B)
LAG_DAYS = [1, 3, 7, 14]
ROLL_WINDOWS = [3, 7, 14]

# Leakage guard keywords
LEAKAGE_KEYWORDS = ["target", "label", "future", "enter5", "trans", "shift", "lead"]

print("Setup complete.")
print("Primary path:", PRIMARY_DATA_PATH)
print("Secondary path:", SECONDARY_DATA_PATH)
print("Target:", TARGET_COL)
print("Group:", GROUP_COL)
print("Order column:", ORDER_COL)
print("Lag days:", LAG_DAYS)
print("Rolling windows:", ROLL_WINDOWS)

Setup complete.
Primary path: DSSAT_Merged_Neat.csv
Secondary path: DSSAT_Final_weather_only_until_2050259.csv
Target: y_enter5_7
Group: RUNNO
Order column: DAP
Lag days: [1, 3, 7, 14]
Rolling windows: [3, 7, 14]


In [4]:
# =========================================================
# SECTION 2: LOAD BOTH DATASETS AND STANDARDIZE COLUMNS
# =========================================================

# Load datasets
df_primary = pd.read_csv(PRIMARY_DATA_PATH)
df_secondary = pd.read_csv(SECONDARY_DATA_PATH)

# Standardize column names
df_primary.columns = [c.strip() for c in df_primary.columns]
df_secondary.columns = [c.strip() for c in df_secondary.columns]

print("Primary raw shape:", df_primary.shape)
print("Secondary raw shape:", df_secondary.shape)

# Required columns for this notebook
required_cols = [GROUP_COL, ORDER_COL, "GSTD"]

missing_primary = [c for c in required_cols if c not in df_primary.columns]
missing_secondary = [c for c in required_cols if c not in df_secondary.columns]

if missing_primary:
    raise ValueError(f"Primary dataset missing required columns: {missing_primary}")
if missing_secondary:
    raise ValueError(f"Secondary dataset missing required columns: {missing_secondary}")

# Sort both datasets consistently
df_primary = df_primary.sort_values([GROUP_COL, ORDER_COL]).reset_index(drop=True)
df_secondary = df_secondary.sort_values([GROUP_COL, ORDER_COL]).reset_index(drop=True)

print("\nPrimary confirmed columns:")
print(f"Unique RUNNO: {df_primary[GROUP_COL].nunique()}")
print(f"Unique TRTNO: {df_primary['TRTNO'].nunique() if 'TRTNO' in df_primary.columns else 'TRTNO not present'}")

print("\nSecondary confirmed columns:")
print(f"Unique RUNNO: {df_secondary[GROUP_COL].nunique()}")
print(f"Unique TRTNO: {df_secondary['TRTNO'].nunique() if 'TRTNO' in df_secondary.columns else 'TRTNO not present'}")

print("\nPrimary preview:")
display(df_primary[[c for c in [GROUP_COL, 'TRTNO', ORDER_COL, 'GSTD'] if c in df_primary.columns]].head(10))

print("\nSecondary preview:")
display(df_secondary[[c for c in [GROUP_COL, 'TRTNO', ORDER_COL, 'GSTD'] if c in df_secondary.columns]].head(10))

Primary raw shape: (5520, 46)
Secondary raw shape: (3450, 39)

Primary confirmed columns:
Unique RUNNO: 40
Unique TRTNO: 8

Secondary confirmed columns:
Unique RUNNO: 25
Unique TRTNO: 5

Primary preview:


,RUNNO,TRTNO,DAP,GSTD
0,1,1,0,0
1,1,1,1,0
2,1,1,2,0
3,1,1,3,0
4,1,1,4,0
5,1,1,5,0
6,1,1,6,0
7,1,1,7,0
8,1,1,8,0
9,1,1,9,0



Secondary preview:


,RUNNO,TRTNO,DAP,GSTD
0,1,1,0,0
1,1,1,1,0
2,1,1,2,0
3,1,1,3,0
4,1,1,4,0
5,1,1,5,0
6,1,1,6,0
7,1,1,7,0
8,1,1,8,0
9,1,1,9,0


In [5]:
# =========================================================
# SECTION 3: CREATE y_enter5_7 IN BOTH DATASETS
# 1 = run-day will enter GSTD 5 within the next 7 days
# 0 = otherwise
# Rule:
# - only look within the same RUNNO
# - if current GSTD is already 5, target = 0
# =========================================================

def create_enter5_target_series(group, horizon_days=7):
    daps = group[ORDER_COL].to_numpy()
    gstd = group["GSTD"].to_numpy()
    target = np.zeros(len(group), dtype=int)

    for i in range(len(group)):
        if gstd[i] == 5:
            target[i] = 0
            continue

        future_mask = (daps > daps[i]) & (daps <= daps[i] + horizon_days)
        target[i] = int(np.any(gstd[future_mask] == 5))

    return pd.Series(target, index=group.index)

# Primary target
df_primary[TARGET_COL] = (
    df_primary.groupby(GROUP_COL, group_keys=False)
    .apply(create_enter5_target_series, horizon_days=7)
    .sort_index()
    .astype(int)
)

# Secondary target
df_secondary[TARGET_COL] = (
    df_secondary.groupby(GROUP_COL, group_keys=False)
    .apply(create_enter5_target_series, horizon_days=7)
    .sort_index()
    .astype(int)
)

print("Primary target distribution:")
print(df_primary[TARGET_COL].value_counts(dropna=False).sort_index())
print("Primary target rate:", round(df_primary[TARGET_COL].mean(), 4))

print("\nSecondary target distribution:")
print(df_secondary[TARGET_COL].value_counts(dropna=False).sort_index())
print("Secondary target rate:", round(df_secondary[TARGET_COL].mean(), 4))

print("\nPrimary GSTD x target:")
display(pd.crosstab(df_primary["GSTD"], df_primary[TARGET_COL], margins=True))

print("\nSecondary GSTD x target:")
display(pd.crosstab(df_secondary["GSTD"], df_secondary[TARGET_COL], margins=True))

Primary target distribution:
y_enter5_7
0    5240
1     280
Name: count, dtype: int64
Primary target rate: 0.0507

Secondary target distribution:
y_enter5_7
0    3275
1     175
Name: count, dtype: int64
Secondary target rate: 0.0507

Primary GSTD x target:


y_enter5_7,0,1,All
GSTD,,,
0,1695,0,1695
1,161,87,248
3,0,193,193
5,1372,0,1372
7,600,0,600
8,1412,0,1412
All,5240,280,5520



Secondary GSTD x target:


y_enter5_7,0,1,All
GSTD,,,
0,1042,0,1042
1,104,54,158
3,0,121,121
5,906,0,906
7,375,0,375
8,848,0,848
All,3275,175,3450


In [6]:
# =========================================================
# SECTION 4: STRUCTURAL CLEANUP + SHARED BASE FEATURE SPACE
# - drop structurally invalid / artifact rows
# - keep LAID
# - remove LAI and the full LAI family
# - keep only shared base variables across both datasets
# - no treatment identifiers, no GSTD, no DAP, no leakage-prone fields
# =========================================================

# -----------------------------
# Structural cleanup
# -----------------------------
required_structural_cols = [GROUP_COL, ORDER_COL, "GSTD", TARGET_COL]

df_primary = df_primary.dropna(subset=required_structural_cols).copy()
df_secondary = df_secondary.dropna(subset=required_structural_cols).copy()

df_primary = df_primary.loc[df_primary[ORDER_COL] > 0].copy()
df_secondary = df_secondary.loc[df_secondary[ORDER_COL] > 0].copy()

df_primary = df_primary.sort_values([GROUP_COL, ORDER_COL]).reset_index(drop=True)
df_secondary = df_secondary.sort_values([GROUP_COL, ORDER_COL]).reset_index(drop=True)

print("Primary shape after structural cleanup:", df_primary.shape)
print("Secondary shape after structural cleanup:", df_secondary.shape)

# -----------------------------
# Exclusion policy
# -----------------------------
explicit_exclude = {
    TARGET_COL,
    "GSTD",
    ORDER_COL,
    GROUP_COL,
    "TRTNO",
    "TRTNA",
    "DOY",
    "WDATE",
    "DATE",
    "ROTNO",
    "REPNO",
    "LAI"  # remove redundant LAI; keep LAID as canonical variable
}

def build_base_feature_list(df):
    merge_artifact_cols = [
        c for c in df.columns
        if c.endswith("_x") or c.endswith("_y") or c.endswith("_WX")
    ]

    auto_leakage_cols = [
        c for c in df.columns
        if c != TARGET_COL and any(k in c.lower() for k in LEAKAGE_KEYWORDS)
    ]

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    feature_cols = [
        c for c in numeric_cols
        if c not in explicit_exclude
        and c not in merge_artifact_cols
        and c not in auto_leakage_cols
        and not (c == "LAI" or c.startswith("LAI_"))
    ]

    return feature_cols, merge_artifact_cols, auto_leakage_cols

primary_base_candidates, primary_merge_artifacts, primary_auto_leakage = build_base_feature_list(df_primary)
secondary_base_candidates, secondary_merge_artifacts, secondary_auto_leakage = build_base_feature_list(df_secondary)

# Shared base features across both datasets
shared_base_feature_cols = sorted(list(set(primary_base_candidates).intersection(set(secondary_base_candidates))))

print("\nPrimary base candidate count:", len(primary_base_candidates))
print("Secondary base candidate count:", len(secondary_base_candidates))
print("Shared base feature count:", len(shared_base_feature_cols))

print("\nShared base features:")
print(shared_base_feature_cols)

print("\nFeatures in primary but not secondary:")
print(sorted(set(primary_base_candidates) - set(shared_base_feature_cols)))

print("\nFeatures in secondary but not primary:")
print(sorted(set(secondary_base_candidates) - set(shared_base_feature_cols)))

# -----------------------------
# Honest missing handling for shared base features
# forward fill only within RUNNO
# -----------------------------
df_primary_base = df_primary.copy()
df_secondary_base = df_secondary.copy()

df_primary_base[shared_base_feature_cols] = (
    df_primary_base.groupby(GROUP_COL)[shared_base_feature_cols]
    .transform(lambda x: x.ffill())
)

df_secondary_base[shared_base_feature_cols] = (
    df_secondary_base.groupby(GROUP_COL)[shared_base_feature_cols]
    .transform(lambda x: x.ffill())
)

print("\nTop missing rates in primary shared base features:")
print(df_primary_base[shared_base_feature_cols].isna().mean().sort_values(ascending=False).head(10))

print("\nTop missing rates in secondary shared base features:")
print(df_secondary_base[shared_base_feature_cols].isna().mean().sort_values(ascending=False).head(10))

Primary shape after structural cleanup: (5480, 47)
Secondary shape after structural cleanup: (3425, 40)

Primary base candidate count: 32
Secondary base candidate count: 32
Shared base feature count: 32

Shared base features:
['CWAD', 'GWAD', 'LAID', 'NI1D', 'NI2D', 'NI3D', 'NI4D', 'NI5D', 'NI6D', 'NI7D', 'NI8D', 'NI9D', 'NIAD', 'NITD', 'PARD', 'SRAD', 'SW10D', 'SW1D', 'SW2D', 'SW3D', 'SW4D', 'SW5D', 'SW6D', 'SW7D', 'SW8D', 'SW9D', 'SWTD', 'SWXD', 'TGRD', 'TMAX', 'TMIN', 'VPD']

Features in primary but not secondary:
[]

Features in secondary but not primary:
[]

Top missing rates in primary shared base features:
CWAD    0.0
GWAD    0.0
LAID    0.0
NI1D    0.0
NI2D    0.0
NI3D    0.0
NI4D    0.0
NI5D    0.0
NI6D    0.0
NI7D    0.0
dtype: float64

Top missing rates in secondary shared base features:
CWAD    0.0
GWAD    0.0
LAID    0.0
NI1D    0.0
NI2D    0.0
NI3D    0.0
NI4D    0.0
NI5D    0.0
NI6D    0.0
NI7D    0.0
dtype: float64


In [7]:
# =========================================================
# SECTION 5: BUILD BASELINE NO-TS DATASETS
# Model A = baseline LightGBM using shared base features only
# =========================================================

# Primary baseline dataset
X_primary_base = df_primary_base[shared_base_feature_cols].copy()
y_primary_base = df_primary_base[TARGET_COL].astype(int).copy()
groups_primary_base = df_primary_base[GROUP_COL].copy()

# Secondary baseline dataset
X_secondary_base = df_secondary_base[shared_base_feature_cols].copy()
y_secondary_base = df_secondary_base[TARGET_COL].astype(int).copy()
groups_secondary_base = df_secondary_base[GROUP_COL].copy()

# Keep only complete rows for the shared base feature space
primary_base_valid_mask = ~X_primary_base.isna().any(axis=1)
secondary_base_valid_mask = ~X_secondary_base.isna().any(axis=1)

X_primary_base_final = X_primary_base.loc[primary_base_valid_mask].copy()
y_primary_base_final = y_primary_base.loc[primary_base_valid_mask].copy()
groups_primary_base_final = groups_primary_base.loc[primary_base_valid_mask].copy()

X_secondary_base_final = X_secondary_base.loc[secondary_base_valid_mask].copy()
y_secondary_base_final = y_secondary_base.loc[secondary_base_valid_mask].copy()
groups_secondary_base_final = groups_secondary_base.loc[secondary_base_valid_mask].copy()

print("Primary baseline shape:", X_primary_base_final.shape)
print("Secondary baseline shape:", X_secondary_base_final.shape)

print("\nPrimary baseline target distribution:")
print(y_primary_base_final.value_counts(dropna=False).sort_index())

print("\nSecondary baseline target distribution:")
print(y_secondary_base_final.value_counts(dropna=False).sort_index())

print("\nPrimary baseline positive rate:", round(y_primary_base_final.mean(), 4))
print("Secondary baseline positive rate:", round(y_secondary_base_final.mean(), 4))

print("\nPrimary baseline unique runs:", groups_primary_base_final.nunique())
print("Secondary baseline unique runs:", groups_secondary_base_final.nunique())

Primary baseline shape: (5480, 32)
Secondary baseline shape: (3425, 32)

Primary baseline target distribution:
y_enter5_7
0    5200
1     280
Name: count, dtype: int64

Secondary baseline target distribution:
y_enter5_7
0    3250
1     175
Name: count, dtype: int64

Primary baseline positive rate: 0.0511
Secondary baseline positive rate: 0.0511

Primary baseline unique runs: 40
Secondary baseline unique runs: 25


In [8]:
# =========================================================
# SECTION 6: PRIMARY GROUPKFOLD EVALUATION FOR BASELINE NO-TS LIGHTGBM
# =========================================================

# GroupKFold for honest evaluation by RUNNO
gkf_base = GroupKFold(n_splits=5)

# More conservative LightGBM setup for stability
baseline_params = {
    "objective": "binary",
    "metric": "None",
    "boosting_type": "gbdt",
    "learning_rate": 0.01,
    "n_estimators": 3000,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 60,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.5,
    "reg_lambda": 2.0,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1
}

# Class weighting based on primary baseline dataset
pos_count_base = int(y_primary_base_final.sum())
neg_count_base = int((y_primary_base_final == 0).sum())
baseline_params["scale_pos_weight"] = neg_count_base / pos_count_base

print("Baseline scale_pos_weight:", round(baseline_params["scale_pos_weight"], 4))

oof_pred_base = np.zeros(len(X_primary_base_final))
baseline_fold_metrics = []
baseline_fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
    gkf_base.split(X_primary_base_final, y_primary_base_final, groups_primary_base_final), start=1
):
    X_train = X_primary_base_final.iloc[train_idx]
    X_valid = X_primary_base_final.iloc[valid_idx]
    y_train = y_primary_base_final.iloc[train_idx]
    y_valid = y_primary_base_final.iloc[valid_idx]

    model = lgb.LGBMClassifier(**baseline_params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=150, verbose=False)]
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]
    oof_pred_base[valid_idx] = valid_pred
    baseline_fold_models.append(model)

    baseline_fold_metrics.append({
        "fold": fold,
        "best_iteration": model.best_iteration_,
        "roc_auc": roc_auc_score(y_valid, valid_pred),
        "pr_auc": average_precision_score(y_valid, valid_pred),
    })

baseline_fold_results = pd.DataFrame(baseline_fold_metrics)

print("\nBaseline no-TS fold results:")
display(baseline_fold_results)

print("\nBaseline no-TS CV summary:")
print("Mean ROC-AUC :", round(baseline_fold_results["roc_auc"].mean(), 4))
print("Std  ROC-AUC :", round(baseline_fold_results["roc_auc"].std(), 4))
print("Mean PR-AUC  :", round(baseline_fold_results["pr_auc"].mean(), 4))
print("Std  PR-AUC  :", round(baseline_fold_results["pr_auc"].std(), 4))

print("\nBaseline no-TS OOF overall:")
print("OOF ROC-AUC :", round(roc_auc_score(y_primary_base_final, oof_pred_base), 4))
print("OOF PR-AUC  :", round(average_precision_score(y_primary_base_final, oof_pred_base), 4))

Baseline scale_pos_weight: 18.5714

Baseline no-TS fold results:


,fold,best_iteration,roc_auc,pr_auc
0,1,8,0.997630,0.917969
1,2,373,1.000000,1.000000
2,3,155,0.999794,0.996388
3,4,62,1.000000,1.000000
4,5,437,0.995570,0.932806



Baseline no-TS CV summary:
Mean ROC-AUC : 0.9986
Std  ROC-AUC : 0.002
Mean PR-AUC  : 0.9694
Std  PR-AUC  : 0.0406

Baseline no-TS OOF overall:
OOF ROC-AUC : 0.9888
OOF PR-AUC  : 0.8597


In [9]:
# =========================================================
# SECTION 7: TRAIN BASELINE NO-TS MODEL ON FULL PRIMARY DATA
# AND EVALUATE ON SECONDARY DATA
# =========================================================

# Train final baseline model on full primary baseline dataset
baseline_final_model = lgb.LGBMClassifier(**baseline_params)
baseline_final_model.fit(X_primary_base_final, y_primary_base_final)

# Predict on secondary baseline dataset
secondary_pred_base = baseline_final_model.predict_proba(X_secondary_base_final)[:, 1]

# Ranking metrics on secondary
secondary_base_roc_auc = roc_auc_score(y_secondary_base_final, secondary_pred_base)
secondary_base_pr_auc = average_precision_score(y_secondary_base_final, secondary_pred_base)

print("Baseline no-TS secondary ranking performance:")
print("Secondary ROC-AUC :", round(secondary_base_roc_auc, 4))
print("Secondary PR-AUC  :", round(secondary_base_pr_auc, 4))

# Threshold scan on secondary for baseline model
secondary_thresholds = np.round(np.arange(0.001, 0.201, 0.001), 3)
baseline_secondary_rows = []

for thr in secondary_thresholds:
    pred_label = (secondary_pred_base >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_secondary_base_final, pred_label).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    alert_rate = pred_label.mean()

    baseline_secondary_rows.append({
        "threshold": thr,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "alert_rate": alert_rate
    })

baseline_secondary_threshold_df = pd.DataFrame(baseline_secondary_rows)

print("\nBaseline no-TS best secondary thresholds by F1:")
display(baseline_secondary_threshold_df.sort_values("f1", ascending=False).head(15))

print("\nBaseline no-TS best secondary thresholds by recall (precision >= 0.20):")
display(
    baseline_secondary_threshold_df.loc[baseline_secondary_threshold_df["precision"] >= 0.20]
    .sort_values(["recall", "precision"], ascending=False)
    .head(15)
)

print("\nBaseline no-TS best secondary thresholds by precision (recall >= 0.20):")
display(
    baseline_secondary_threshold_df.loc[baseline_secondary_threshold_df["recall"] >= 0.20]
    .sort_values(["precision", "recall"], ascending=False)
    .head(15)
)

Baseline no-TS secondary ranking performance:
Secondary ROC-AUC : 0.7359
Secondary PR-AUC  : 0.1092

Baseline no-TS best secondary thresholds by F1:


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
0,0.001,168,1221,2029,7,0.120950,0.960000,0.624308,0.214834,0.405547
1,0.002,138,1187,2063,37,0.104151,0.788571,0.634769,0.184000,0.386861
2,0.003,124,1167,2083,51,0.096050,0.708571,0.640923,0.169168,0.376934
3,0.004,116,1162,2088,59,0.090767,0.662857,0.642462,0.159670,0.373139
4,0.005,113,1135,2115,62,0.090545,0.645714,0.650769,0.158819,0.364380
5,0.006,109,1112,2138,66,0.089271,0.622857,0.657846,0.156160,0.356496
6,0.007,104,1072,2178,71,0.088435,0.594286,0.670154,0.153960,0.343358
7,0.008,98,1035,2215,77,0.086496,0.560000,0.681538,0.149847,0.330803
8,0.009,94,1011,2239,81,0.085068,0.537143,0.688923,0.146875,0.322628
9,0.010,87,985,2265,88,0.081157,0.497143,0.696923,0.139535,0.312993



Baseline no-TS best secondary thresholds by recall (precision >= 0.20):


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate



Baseline no-TS best secondary thresholds by precision (recall >= 0.20):


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
0,0.001,168,1221,2029,7,0.120950,0.960000,0.624308,0.214834,0.405547
1,0.002,138,1187,2063,37,0.104151,0.788571,0.634769,0.184000,0.386861
2,0.003,124,1167,2083,51,0.096050,0.708571,0.640923,0.169168,0.376934
3,0.004,116,1162,2088,59,0.090767,0.662857,0.642462,0.159670,0.373139
4,0.005,113,1135,2115,62,0.090545,0.645714,0.650769,0.158819,0.364380
5,0.006,109,1112,2138,66,0.089271,0.622857,0.657846,0.156160,0.356496
6,0.007,104,1072,2178,71,0.088435,0.594286,0.670154,0.153960,0.343358
7,0.008,98,1035,2215,77,0.086496,0.560000,0.681538,0.149847,0.330803
8,0.009,94,1011,2239,81,0.085068,0.537143,0.688923,0.146875,0.322628
10,0.011,85,962,2288,90,0.081184,0.485714,0.704000,0.139116,0.305693


In [10]:
# =========================================================
# SECTION 8: BUILD TS-ENGINEERED SHARED-FEATURE DATASETS
# Model B = LightGBM with lags, rolling means, and first differences
# using only the shared base feature space
# =========================================================

# Start from the shared-base versions
df_primary_ts = df_primary_base.copy()
df_secondary_ts = df_secondary_base.copy()

# Engineer TS features on primary
for col in shared_base_feature_cols:
    grouped = df_primary_ts.groupby(GROUP_COL)[col]

    for lag in LAG_DAYS:
        df_primary_ts[f"{col}_lag{lag}"] = grouped.shift(lag)

    for win in ROLL_WINDOWS:
        df_primary_ts[f"{col}_rollmean{win}"] = (
            grouped.transform(lambda s: s.rolling(window=win, min_periods=win).mean())
        )

    df_primary_ts[f"{col}_diff1"] = grouped.diff(1)

# Engineer TS features on secondary
for col in shared_base_feature_cols:
    grouped = df_secondary_ts.groupby(GROUP_COL)[col]

    for lag in LAG_DAYS:
        df_secondary_ts[f"{col}_lag{lag}"] = grouped.shift(lag)

    for win in ROLL_WINDOWS:
        df_secondary_ts[f"{col}_rollmean{win}"] = (
            grouped.transform(lambda s: s.rolling(window=win, min_periods=win).mean())
        )

    df_secondary_ts[f"{col}_diff1"] = grouped.diff(1)

# Final shared TS feature list
ts_feature_cols = [
    c for c in df_primary_ts.columns
    if c in df_secondary_ts.columns
    and (
        c in shared_base_feature_cols
        or "_lag" in c
        or "_rollmean" in c
        or c.endswith("_diff1")
    )
]

ts_feature_cols = sorted(ts_feature_cols)

# Build primary TS matrices
X_primary_ts = df_primary_ts[ts_feature_cols].copy()
y_primary_ts = df_primary_ts[TARGET_COL].astype(int).copy()
groups_primary_ts = df_primary_ts[GROUP_COL].copy()

# Build secondary TS matrices
X_secondary_ts = df_secondary_ts[ts_feature_cols].copy()
y_secondary_ts = df_secondary_ts[TARGET_COL].astype(int).copy()
groups_secondary_ts = df_secondary_ts[GROUP_COL].copy()

# Drop incomplete-history rows
primary_ts_valid_mask = ~X_primary_ts.isna().any(axis=1)
secondary_ts_valid_mask = ~X_secondary_ts.isna().any(axis=1)

X_primary_ts_final = X_primary_ts.loc[primary_ts_valid_mask].copy()
y_primary_ts_final = y_primary_ts.loc[primary_ts_valid_mask].copy()
groups_primary_ts_final = groups_primary_ts.loc[primary_ts_valid_mask].copy()

X_secondary_ts_final = X_secondary_ts.loc[secondary_ts_valid_mask].copy()
y_secondary_ts_final = y_secondary_ts.loc[secondary_ts_valid_mask].copy()
groups_secondary_ts_final = groups_secondary_ts.loc[secondary_ts_valid_mask].copy()

print("Shared TS feature count:", len(ts_feature_cols))

print("\nPrimary TS shape:", X_primary_ts_final.shape)
print("Secondary TS shape:", X_secondary_ts_final.shape)

print("\nPrimary TS target distribution:")
print(y_primary_ts_final.value_counts(dropna=False).sort_index())

print("\nSecondary TS target distribution:")
print(y_secondary_ts_final.value_counts(dropna=False).sort_index())

print("\nPrimary TS positive rate:", round(y_primary_ts_final.mean(), 4))
print("Secondary TS positive rate:", round(y_secondary_ts_final.mean(), 4))

print("\nPrimary TS unique runs:", groups_primary_ts_final.nunique())
print("Secondary TS unique runs:", groups_secondary_ts_final.nunique())

Shared TS feature count: 288

Primary TS shape: (4920, 288)
Secondary TS shape: (3075, 288)

Primary TS target distribution:
y_enter5_7
0    4640
1     280
Name: count, dtype: int64

Secondary TS target distribution:
y_enter5_7
0    2900
1     175
Name: count, dtype: int64

Primary TS positive rate: 0.0569
Secondary TS positive rate: 0.0569

Primary TS unique runs: 40
Secondary TS unique runs: 25


In [11]:
# =========================================================
# SECTION 9: PRIMARY GROUPKFOLD EVALUATION FOR TS-ENGINEERED LIGHTGBM
# =========================================================

# GroupKFold for honest evaluation by RUNNO
gkf_ts = GroupKFold(n_splits=5)

# Use the same conservative LightGBM settings
ts_params = {
    "objective": "binary",
    "metric": "None",
    "boosting_type": "gbdt",
    "learning_rate": 0.01,
    "n_estimators": 3000,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 60,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.5,
    "reg_lambda": 2.0,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1
}

# Class weighting based on primary TS dataset
pos_count_ts = int(y_primary_ts_final.sum())
neg_count_ts = int((y_primary_ts_final == 0).sum())
ts_params["scale_pos_weight"] = neg_count_ts / pos_count_ts

print("TS scale_pos_weight:", round(ts_params["scale_pos_weight"], 4))

oof_pred_ts = np.zeros(len(X_primary_ts_final))
ts_fold_metrics = []
ts_fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
    gkf_ts.split(X_primary_ts_final, y_primary_ts_final, groups_primary_ts_final), start=1
):
    X_train = X_primary_ts_final.iloc[train_idx]
    X_valid = X_primary_ts_final.iloc[valid_idx]
    y_train = y_primary_ts_final.iloc[train_idx]
    y_valid = y_primary_ts_final.iloc[valid_idx]

    model = lgb.LGBMClassifier(**ts_params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=150, verbose=False)]
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]
    oof_pred_ts[valid_idx] = valid_pred
    ts_fold_models.append(model)

    ts_fold_metrics.append({
        "fold": fold,
        "best_iteration": model.best_iteration_,
        "roc_auc": roc_auc_score(y_valid, valid_pred),
        "pr_auc": average_precision_score(y_valid, valid_pred),
    })

ts_fold_results = pd.DataFrame(ts_fold_metrics)

print("\nTS-engineered fold results:")
display(ts_fold_results)

print("\nTS-engineered CV summary:")
print("Mean ROC-AUC :", round(ts_fold_results["roc_auc"].mean(), 4))
print("Std  ROC-AUC :", round(ts_fold_results["roc_auc"].std(), 4))
print("Mean PR-AUC  :", round(ts_fold_results["pr_auc"].mean(), 4))
print("Std  PR-AUC  :", round(ts_fold_results["pr_auc"].std(), 4))

print("\nTS-engineered OOF overall:")
print("OOF ROC-AUC :", round(roc_auc_score(y_primary_ts_final, oof_pred_ts), 4))
print("OOF PR-AUC  :", round(average_precision_score(y_primary_ts_final, oof_pred_ts), 4))

TS scale_pos_weight: 16.5714

TS-engineered fold results:


,fold,best_iteration,roc_auc,pr_auc
0,1,54,1.000000,1.000000
1,2,300,1.000000,1.000000
2,3,25,1.000000,1.000000
3,4,25,1.000000,1.000000
4,5,198,0.997402,0.942032



TS-engineered CV summary:
Mean ROC-AUC : 0.9995
Std  ROC-AUC : 0.0012
Mean PR-AUC  : 0.9884
Std  PR-AUC  : 0.0259

TS-engineered OOF overall:
OOF ROC-AUC : 0.996
OOF PR-AUC  : 0.9092


In [12]:
# =========================================================
# SECTION 10: TRAIN TS-ENGINEERED MODEL ON FULL PRIMARY DATA
# AND EVALUATE ON SECONDARY DATA
# =========================================================

# Train final TS-engineered model on full primary TS dataset
ts_final_model = lgb.LGBMClassifier(**ts_params)
ts_final_model.fit(X_primary_ts_final, y_primary_ts_final)

# Predict on secondary TS dataset
secondary_pred_ts = ts_final_model.predict_proba(X_secondary_ts_final)[:, 1]

# Ranking metrics on secondary
secondary_ts_roc_auc = roc_auc_score(y_secondary_ts_final, secondary_pred_ts)
secondary_ts_pr_auc = average_precision_score(y_secondary_ts_final, secondary_pred_ts)

print("TS-engineered secondary ranking performance:")
print("Secondary ROC-AUC :", round(secondary_ts_roc_auc, 4))
print("Secondary PR-AUC  :", round(secondary_ts_pr_auc, 4))

# Threshold scan on secondary for TS model
secondary_thresholds = np.round(np.arange(0.001, 0.201, 0.001), 3)
ts_secondary_rows = []

for thr in secondary_thresholds:
    pred_label = (secondary_pred_ts >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_secondary_ts_final, pred_label).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    alert_rate = pred_label.mean()

    ts_secondary_rows.append({
        "threshold": thr,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "alert_rate": alert_rate
    })

ts_secondary_threshold_df = pd.DataFrame(ts_secondary_rows)

print("\nTS-engineered best secondary thresholds by F1:")
display(ts_secondary_threshold_df.sort_values("f1", ascending=False).head(15))

print("\nTS-engineered best secondary thresholds by recall (precision >= 0.20):")
display(
    ts_secondary_threshold_df.loc[ts_secondary_threshold_df["precision"] >= 0.20]
    .sort_values(["recall", "precision"], ascending=False)
    .head(15)
)

print("\nTS-engineered best secondary thresholds by precision (recall >= 0.20):")
display(
    ts_secondary_threshold_df.loc[ts_secondary_threshold_df["recall"] >= 0.20]
    .sort_values(["precision", "recall"], ascending=False)
    .head(15)
)

TS-engineered secondary ranking performance:
Secondary ROC-AUC : 0.8399
Secondary PR-AUC  : 0.1574

TS-engineered best secondary thresholds by F1:


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
1,0.002,164,749,2151,11,0.179628,0.937143,0.741724,0.301471,0.296911
0,0.001,175,833,2067,0,0.173611,1.000000,0.712759,0.295858,0.327805
2,0.003,125,634,2266,50,0.164690,0.714286,0.781379,0.267666,0.246829
3,0.004,101,537,2363,74,0.158307,0.577143,0.814828,0.248462,0.207480
4,0.005,79,462,2438,96,0.146026,0.451429,0.840690,0.220670,0.175935
5,0.006,67,396,2504,108,0.144708,0.382857,0.863448,0.210031,0.150569
6,0.007,57,349,2551,118,0.140394,0.325714,0.879655,0.196213,0.132033
7,0.008,47,315,2585,128,0.129834,0.268571,0.891379,0.175047,0.117724
8,0.009,38,284,2616,137,0.118012,0.217143,0.902069,0.152918,0.104715
9,0.010,33,239,2661,142,0.121324,0.188571,0.917586,0.147651,0.088455



TS-engineered best secondary thresholds by recall (precision >= 0.20):


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
69,0.070,6,24,2876,169,0.200000,0.034286,0.991724,0.058537,0.009756
70,0.071,6,24,2876,169,0.200000,0.034286,0.991724,0.058537,0.009756
71,0.072,6,24,2876,169,0.200000,0.034286,0.991724,0.058537,0.009756
72,0.073,6,24,2876,169,0.200000,0.034286,0.991724,0.058537,0.009756
73,0.074,6,24,2876,169,0.200000,0.034286,0.991724,0.058537,0.009756
110,0.111,5,10,2890,170,0.333333,0.028571,0.996552,0.052632,0.004878
111,0.112,5,10,2890,170,0.333333,0.028571,0.996552,0.052632,0.004878
108,0.109,5,11,2889,170,0.312500,0.028571,0.996207,0.052356,0.005203
109,0.110,5,11,2889,170,0.312500,0.028571,0.996207,0.052356,0.005203
104,0.105,5,12,2888,170,0.294118,0.028571,0.995862,0.052083,0.005528



TS-engineered best secondary thresholds by precision (recall >= 0.20):


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
1,0.002,164,749,2151,11,0.179628,0.937143,0.741724,0.301471,0.296911
0,0.001,175,833,2067,0,0.173611,1.000000,0.712759,0.295858,0.327805
2,0.003,125,634,2266,50,0.164690,0.714286,0.781379,0.267666,0.246829
3,0.004,101,537,2363,74,0.158307,0.577143,0.814828,0.248462,0.207480
4,0.005,79,462,2438,96,0.146026,0.451429,0.840690,0.220670,0.175935
5,0.006,67,396,2504,108,0.144708,0.382857,0.863448,0.210031,0.150569
6,0.007,57,349,2551,118,0.140394,0.325714,0.879655,0.196213,0.132033
7,0.008,47,315,2585,128,0.129834,0.268571,0.891379,0.175047,0.117724
8,0.009,38,284,2616,137,0.118012,0.217143,0.902069,0.152918,0.104715


In [13]:
# =========================================================
# SECTION 11: COMPARE BASELINE NO-TS VS TS-ENGINEERED MODELS
# =========================================================

comparison_summary = pd.DataFrame([
    {
        "model": "Baseline LightGBM (no TS)",
        "primary_oof_roc_auc": roc_auc_score(y_primary_base_final, oof_pred_base),
        "primary_oof_pr_auc": average_precision_score(y_primary_base_final, oof_pred_base),
        "secondary_roc_auc": secondary_base_roc_auc,
        "secondary_pr_auc": secondary_base_pr_auc,
        "secondary_best_f1": baseline_secondary_threshold_df["f1"].max(),
        "secondary_best_f1_threshold": baseline_secondary_threshold_df.loc[
            baseline_secondary_threshold_df["f1"].idxmax(), "threshold"
        ],
    },
    {
        "model": "TS-engineered LightGBM",
        "primary_oof_roc_auc": roc_auc_score(y_primary_ts_final, oof_pred_ts),
        "primary_oof_pr_auc": average_precision_score(y_primary_ts_final, oof_pred_ts),
        "secondary_roc_auc": secondary_ts_roc_auc,
        "secondary_pr_auc": secondary_ts_pr_auc,
        "secondary_best_f1": ts_secondary_threshold_df["f1"].max(),
        "secondary_best_f1_threshold": ts_secondary_threshold_df.loc[
            ts_secondary_threshold_df["f1"].idxmax(), "threshold"
        ],
    }
])

print("Model comparison summary:")
display(comparison_summary)

# Determine which model is better on the secondary dataset
best_secondary_model = comparison_summary.sort_values(
    ["secondary_pr_auc", "secondary_roc_auc"], ascending=False
).iloc[0]["model"]

print("Best model on secondary robustness data:", best_secondary_model)

print("\nInterpretation:")
print(
    "Primary OOF metrics show internal grouped-validation performance on the primary dataset. "
    "Secondary metrics show how well each model transfers to the separate robustness dataset."
)

Model comparison summary:


,model,primary_oof_roc_auc,primary_oof_pr_auc,secondary_roc_auc,secondary_pr_auc,secondary_best_f1,secondary_best_f1_threshold
0,Baseline LightGBM (no TS),0.988782,0.859720,0.735889,0.109228,0.214834,0.001
1,TS-engineered LightGBM,0.996010,0.909153,0.839870,0.157372,0.301471,0.002


Best model on secondary robustness data: TS-engineered LightGBM

Interpretation:
Primary OOF metrics show internal grouped-validation performance on the primary dataset. Secondary metrics show how well each model transfers to the separate robustness dataset.


Training portion shape: (3936, 288)
Calibration portion shape: (984, 288)

Training target distribution:
y_enter5_7
0    3712
1     224
Name: count, dtype: int64

Calibration target distribution:
y_enter5_7
0    928
1     56
Name: count, dtype: int64

Training unique runs: 32
Calibration unique runs: 8

Training positive rate: 0.0569
Calibration positive rate: 0.0569


In [16]:
# =========================================================
# SECTION 13: FIT UNCALIBRATED, SIGMOID, AND ISOTONIC VERSIONS
# Train TS model on primary-train split and calibrate on held-out primary runs
# =========================================================

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

# Train the base TS model on the training portion
ts_calibration_params = ts_params.copy()

pos_count_train = int(y_train_cal.sum())
neg_count_train = int((y_train_cal == 0).sum())
ts_calibration_params["scale_pos_weight"] = neg_count_train / pos_count_train

ts_model_for_calibration = lgb.LGBMClassifier(**ts_calibration_params)
ts_model_for_calibration.fit(X_train_cal, y_train_cal)

# Raw probabilities on the held-out calibration portion
calib_raw_pred = ts_model_for_calibration.predict_proba(X_holdout_cal)[:, 1]

print("Uncalibrated calibration-split performance:")
print("ROC-AUC :", round(roc_auc_score(y_holdout_cal, calib_raw_pred), 4))
print("PR-AUC  :", round(average_precision_score(y_holdout_cal, calib_raw_pred), 4))

# -----------------------------
# Sigmoid / Platt scaling
# -----------------------------
sigmoid_calibrator = LogisticRegression(random_state=SEED)
sigmoid_calibrator.fit(calib_raw_pred.reshape(-1, 1), y_holdout_cal)

calib_sigmoid_pred = sigmoid_calibrator.predict_proba(calib_raw_pred.reshape(-1, 1))[:, 1]

print("\nSigmoid-calibrated calibration-split performance:")
print("ROC-AUC :", round(roc_auc_score(y_holdout_cal, calib_sigmoid_pred), 4))
print("PR-AUC  :", round(average_precision_score(y_holdout_cal, calib_sigmoid_pred), 4))

# -----------------------------
# Isotonic calibration
# -----------------------------
isotonic_calibrator = IsotonicRegression(out_of_bounds="clip")
isotonic_calibrator.fit(calib_raw_pred, y_holdout_cal)

calib_isotonic_pred = isotonic_calibrator.predict(calib_raw_pred)

print("\nIsotonic-calibrated calibration-split performance:")
print("ROC-AUC :", round(roc_auc_score(y_holdout_cal, calib_isotonic_pred), 4))
print("PR-AUC  :", round(average_precision_score(y_holdout_cal, calib_isotonic_pred), 4))

# Quick probability range sanity check
probability_range_summary = pd.DataFrame({
    "version": ["uncalibrated", "sigmoid", "isotonic"],
    "min_pred": [calib_raw_pred.min(), calib_sigmoid_pred.min(), calib_isotonic_pred.min()],
    "max_pred": [calib_raw_pred.max(), calib_sigmoid_pred.max(), calib_isotonic_pred.max()],
    "mean_pred": [calib_raw_pred.mean(), calib_sigmoid_pred.mean(), calib_isotonic_pred.mean()]
})

print("\nCalibration probability range summary:")
display(probability_range_summary)

Uncalibrated calibration-split performance:
ROC-AUC : 1.0
PR-AUC  : 1.0

Sigmoid-calibrated calibration-split performance:
ROC-AUC : 1.0
PR-AUC  : 1.0

Isotonic-calibrated calibration-split performance:
ROC-AUC : 1.0
PR-AUC  : 1.0

Calibration probability range summary:


,version,min_pred,max_pred,mean_pred
0,uncalibrated,0.000010,0.999924,0.057121
1,sigmoid,0.007391,0.877913,0.056938
2,isotonic,0.000000,1.000000,0.056911


In [17]:
# =========================================================
# SECTION 14: APPLY UNCALIBRATED + CALIBRATED VERSIONS TO SECONDARY
# Compare threshold behavior on the secondary dataset
# =========================================================

# Use the already-trained TS model from the calibration workflow
secondary_raw_pred_cal = ts_model_for_calibration.predict_proba(X_secondary_ts_final)[:, 1]

# Apply calibrators
secondary_sigmoid_pred = sigmoid_calibrator.predict_proba(
    secondary_raw_pred_cal.reshape(-1, 1)
)[:, 1]

secondary_isotonic_pred = isotonic_calibrator.predict(secondary_raw_pred_cal)

# Ranking metrics comparison
calibration_transfer_summary = pd.DataFrame([
    {
        "version": "uncalibrated",
        "secondary_roc_auc": roc_auc_score(y_secondary_ts_final, secondary_raw_pred_cal),
        "secondary_pr_auc": average_precision_score(y_secondary_ts_final, secondary_raw_pred_cal),
        "min_pred": secondary_raw_pred_cal.min(),
        "max_pred": secondary_raw_pred_cal.max(),
        "mean_pred": secondary_raw_pred_cal.mean()
    },
    {
        "version": "sigmoid",
        "secondary_roc_auc": roc_auc_score(y_secondary_ts_final, secondary_sigmoid_pred),
        "secondary_pr_auc": average_precision_score(y_secondary_ts_final, secondary_sigmoid_pred),
        "min_pred": secondary_sigmoid_pred.min(),
        "max_pred": secondary_sigmoid_pred.max(),
        "mean_pred": secondary_sigmoid_pred.mean()
    },
    {
        "version": "isotonic",
        "secondary_roc_auc": roc_auc_score(y_secondary_ts_final, secondary_isotonic_pred),
        "secondary_pr_auc": average_precision_score(y_secondary_ts_final, secondary_isotonic_pred),
        "min_pred": secondary_isotonic_pred.min(),
        "max_pred": secondary_isotonic_pred.max(),
        "mean_pred": secondary_isotonic_pred.mean()
    }
])

print("Secondary ranking and probability-scale comparison:")
display(calibration_transfer_summary)

# Threshold scan helper
def threshold_scan(y_true, pred_proba, thresholds):
    rows = []
    for thr in thresholds:
        pred_label = (pred_proba >= thr).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, pred_label).ravel()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        alert_rate = pred_label.mean()

        rows.append({
            "threshold": thr,
            "tp": int(tp),
            "fp": int(fp),
            "tn": int(tn),
            "fn": int(fn),
            "precision": precision,
            "recall": recall,
            "specificity": specificity,
            "f1": f1,
            "alert_rate": alert_rate
        })
    return pd.DataFrame(rows)

secondary_thresholds_cal = np.round(np.arange(0.001, 0.201, 0.001), 3)

uncal_df = threshold_scan(y_secondary_ts_final, secondary_raw_pred_cal, secondary_thresholds_cal)
sigmoid_df = threshold_scan(y_secondary_ts_final, secondary_sigmoid_pred, secondary_thresholds_cal)
isotonic_df = threshold_scan(y_secondary_ts_final, secondary_isotonic_pred, secondary_thresholds_cal)

print("\nBest secondary thresholds by F1 — uncalibrated:")
display(uncal_df.sort_values("f1", ascending=False).head(10))

print("\nBest secondary thresholds by F1 — sigmoid:")
display(sigmoid_df.sort_values("f1", ascending=False).head(10))

print("\nBest secondary thresholds by F1 — isotonic:")
display(isotonic_df.sort_values("f1", ascending=False).head(10))

Secondary ranking and probability-scale comparison:


,version,secondary_roc_auc,secondary_pr_auc,min_pred,max_pred,mean_pred
0,uncalibrated,0.838418,0.148547,0.000021,0.333499,0.005496
1,sigmoid,0.838418,0.148547,0.007392,0.068637,0.007778
2,isotonic,0.535413,0.065477,0.000000,0.324425,0.002837



Best secondary thresholds by F1 — uncalibrated:


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
1,0.002,175,799,2101,0,0.179671,1.000000,0.724483,0.304613,0.316748
0,0.001,175,839,2061,0,0.172584,1.000000,0.710690,0.294365,0.329756
2,0.003,156,732,2168,19,0.175676,0.891429,0.747586,0.293509,0.288780
3,0.004,123,643,2257,52,0.160574,0.702857,0.778276,0.261424,0.249106
4,0.005,102,547,2353,73,0.157165,0.582857,0.811379,0.247573,0.211057
5,0.006,83,482,2418,92,0.146903,0.474286,0.833793,0.224324,0.183740
7,0.008,64,381,2519,111,0.143820,0.365714,0.868621,0.206452,0.144715
6,0.007,66,425,2475,109,0.134420,0.377143,0.853448,0.198198,0.159675
8,0.009,54,334,2566,121,0.139175,0.308571,0.884828,0.191829,0.126179
10,0.011,38,263,2637,137,0.126246,0.217143,0.909310,0.159664,0.097886



Best secondary thresholds by F1 — sigmoid:


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
7,0.008,33,242,2658,142,0.120000,0.188571,0.916552,0.146667,0.089431
0,0.001,175,2900,0,0,0.056911,1.000000,0.000000,0.107692,1.000000
2,0.003,175,2900,0,0,0.056911,1.000000,0.000000,0.107692,1.000000
1,0.002,175,2900,0,0,0.056911,1.000000,0.000000,0.107692,1.000000
3,0.004,175,2900,0,0,0.056911,1.000000,0.000000,0.107692,1.000000
4,0.005,175,2900,0,0,0.056911,1.000000,0.000000,0.107692,1.000000
5,0.006,175,2900,0,0,0.056911,1.000000,0.000000,0.107692,1.000000
6,0.007,175,2900,0,0,0.056911,1.000000,0.000000,0.107692,1.000000
10,0.011,9,55,2845,166,0.140625,0.051429,0.981034,0.075314,0.020813
9,0.010,9,67,2833,166,0.118421,0.051429,0.976897,0.071713,0.024715



Best secondary thresholds by F1 — isotonic:


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
0,0.001,23,198,2702,152,0.104072,0.131429,0.931724,0.116162,0.071870
1,0.002,21,184,2716,154,0.102439,0.120000,0.936552,0.110526,0.066667
2,0.003,20,173,2727,155,0.103627,0.114286,0.940345,0.108696,0.062764
3,0.004,17,162,2738,158,0.094972,0.097143,0.944138,0.096045,0.058211
4,0.005,16,156,2744,159,0.093023,0.091429,0.946207,0.092219,0.055935
5,0.006,14,146,2754,161,0.087500,0.080000,0.949655,0.083582,0.052033
6,0.007,13,138,2762,162,0.086093,0.074286,0.952414,0.079755,0.049106
45,0.046,9,55,2845,166,0.140625,0.051429,0.981034,0.075314,0.020813
44,0.045,9,57,2843,166,0.136364,0.051429,0.980345,0.074689,0.021463
7,0.008,12,135,2765,163,0.081633,0.068571,0.953448,0.074534,0.047805


## Calibration Follow-Up Summary

I tested whether probability calibration could improve how well the TS-engineered LightGBM transfers to the secondary dataset.

I compared three versions of the same model:

1. Uncalibrated
2. Sigmoid-calibrated
3. Isotonic-calibrated

The results showed that calibration did **not** improve transfer to the secondary dataset.

- The **uncalibrated** model still gave the best overall secondary performance.
- **Sigmoid calibration** kept the ranking performance about the same, but it did not improve threshold behavior enough to be useful.
- **Isotonic calibration** performed much worse on the secondary dataset and reduced robustness.

This means the main problem is not just probability calibration. The bigger issue is that the two datasets are still different enough that the score scale shifts between them.

Because of that, the best current choice is still the **uncalibrated TS-engineered LightGBM** using the shared feature space across both datasets.

At this stage, the model is best used as a **risk-ranking or prioritization tool** rather than a fixed-threshold alert system across all datasets. Future improvement should focus more on reducing dataset shift, simplifying the feature set, or recalibrating thresholds separately for each deployment environment.

In [18]:
# =========================================================
# SECTION 15: BUILD REDUCED STABILITY-FOCUSED TS DATASETS
# Keep:
# - shared base features
# - lag1
# - lag7
# - diff1
# Remove:
# - lag3
# - lag14
# - all rolling means
# =========================================================

# Start from the shared-base versions
df_primary_reduced_ts = df_primary_base.copy()
df_secondary_reduced_ts = df_secondary_base.copy()

# Engineer reduced TS features on primary
for col in shared_base_feature_cols:
    grouped = df_primary_reduced_ts.groupby(GROUP_COL)[col]

    df_primary_reduced_ts[f"{col}_lag1"] = grouped.shift(1)
    df_primary_reduced_ts[f"{col}_lag7"] = grouped.shift(7)
    df_primary_reduced_ts[f"{col}_diff1"] = grouped.diff(1)

# Engineer reduced TS features on secondary
for col in shared_base_feature_cols:
    grouped = df_secondary_reduced_ts.groupby(GROUP_COL)[col]

    df_secondary_reduced_ts[f"{col}_lag1"] = grouped.shift(1)
    df_secondary_reduced_ts[f"{col}_lag7"] = grouped.shift(7)
    df_secondary_reduced_ts[f"{col}_diff1"] = grouped.diff(1)

# Final reduced TS feature list
reduced_ts_feature_cols = [
    c for c in df_primary_reduced_ts.columns
    if c in df_secondary_reduced_ts.columns
    and (
        c in shared_base_feature_cols
        or c.endswith("_lag1")
        or c.endswith("_lag7")
        or c.endswith("_diff1")
    )
]

reduced_ts_feature_cols = sorted(reduced_ts_feature_cols)

# Build primary reduced TS matrices
X_primary_reduced_ts = df_primary_reduced_ts[reduced_ts_feature_cols].copy()
y_primary_reduced_ts = df_primary_reduced_ts[TARGET_COL].astype(int).copy()
groups_primary_reduced_ts = df_primary_reduced_ts[GROUP_COL].copy()

# Build secondary reduced TS matrices
X_secondary_reduced_ts = df_secondary_reduced_ts[reduced_ts_feature_cols].copy()
y_secondary_reduced_ts = df_secondary_reduced_ts[TARGET_COL].astype(int).copy()
groups_secondary_reduced_ts = df_secondary_reduced_ts[GROUP_COL].copy()

# Drop incomplete-history rows
primary_reduced_valid_mask = ~X_primary_reduced_ts.isna().any(axis=1)
secondary_reduced_valid_mask = ~X_secondary_reduced_ts.isna().any(axis=1)

X_primary_reduced_ts_final = X_primary_reduced_ts.loc[primary_reduced_valid_mask].copy()
y_primary_reduced_ts_final = y_primary_reduced_ts.loc[primary_reduced_valid_mask].copy()
groups_primary_reduced_ts_final = groups_primary_reduced_ts.loc[primary_reduced_valid_mask].copy()

X_secondary_reduced_ts_final = X_secondary_reduced_ts.loc[secondary_reduced_valid_mask].copy()
y_secondary_reduced_ts_final = y_secondary_reduced_ts.loc[secondary_reduced_valid_mask].copy()
groups_secondary_reduced_ts_final = groups_secondary_reduced_ts.loc[secondary_reduced_valid_mask].copy()

print("Reduced TS feature count:", len(reduced_ts_feature_cols))

print("\nPrimary reduced TS shape:", X_primary_reduced_ts_final.shape)
print("Secondary reduced TS shape:", X_secondary_reduced_ts_final.shape)

print("\nPrimary reduced TS target distribution:")
print(y_primary_reduced_ts_final.value_counts(dropna=False).sort_index())

print("\nSecondary reduced TS target distribution:")
print(y_secondary_reduced_ts_final.value_counts(dropna=False).sort_index())

print("\nPrimary reduced TS positive rate:", round(y_primary_reduced_ts_final.mean(), 4))
print("Secondary reduced TS positive rate:", round(y_secondary_reduced_ts_final.mean(), 4))

print("\nPrimary reduced TS unique runs:", groups_primary_reduced_ts_final.nunique())
print("Secondary reduced TS unique runs:", groups_secondary_reduced_ts_final.nunique())

Reduced TS feature count: 128

Primary reduced TS shape: (5200, 128)
Secondary reduced TS shape: (3250, 128)

Primary reduced TS target distribution:
y_enter5_7
0    4920
1     280
Name: count, dtype: int64

Secondary reduced TS target distribution:
y_enter5_7
0    3075
1     175
Name: count, dtype: int64

Primary reduced TS positive rate: 0.0538
Secondary reduced TS positive rate: 0.0538

Primary reduced TS unique runs: 40
Secondary reduced TS unique runs: 25


In [19]:
# =========================================================
# SECTION 16: PRIMARY GROUPKFOLD EVALUATION FOR REDUCED TS MODEL
# =========================================================

gkf_reduced = GroupKFold(n_splits=5)

reduced_ts_params = {
    "objective": "binary",
    "metric": "None",
    "boosting_type": "gbdt",
    "learning_rate": 0.01,
    "n_estimators": 3000,
    "num_leaves": 15,
    "max_depth": 5,
    "min_child_samples": 60,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.5,
    "reg_lambda": 2.0,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1
}

pos_count_reduced = int(y_primary_reduced_ts_final.sum())
neg_count_reduced = int((y_primary_reduced_ts_final == 0).sum())
reduced_ts_params["scale_pos_weight"] = neg_count_reduced / pos_count_reduced

print("Reduced TS scale_pos_weight:", round(reduced_ts_params["scale_pos_weight"], 4))

oof_pred_reduced_ts = np.zeros(len(X_primary_reduced_ts_final))
reduced_ts_fold_metrics = []
reduced_ts_fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
    gkf_reduced.split(X_primary_reduced_ts_final, y_primary_reduced_ts_final, groups_primary_reduced_ts_final), start=1
):
    X_train = X_primary_reduced_ts_final.iloc[train_idx]
    X_valid = X_primary_reduced_ts_final.iloc[valid_idx]
    y_train = y_primary_reduced_ts_final.iloc[train_idx]
    y_valid = y_primary_reduced_ts_final.iloc[valid_idx]

    model = lgb.LGBMClassifier(**reduced_ts_params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=150, verbose=False)]
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]
    oof_pred_reduced_ts[valid_idx] = valid_pred
    reduced_ts_fold_models.append(model)

    reduced_ts_fold_metrics.append({
        "fold": fold,
        "best_iteration": model.best_iteration_,
        "roc_auc": roc_auc_score(y_valid, valid_pred),
        "pr_auc": average_precision_score(y_valid, valid_pred),
    })

reduced_ts_fold_results = pd.DataFrame(reduced_ts_fold_metrics)

print("\nReduced TS fold results:")
display(reduced_ts_fold_results)

print("\nReduced TS CV summary:")
print("Mean ROC-AUC :", round(reduced_ts_fold_results["roc_auc"].mean(), 4))
print("Std  ROC-AUC :", round(reduced_ts_fold_results["roc_auc"].std(), 4))
print("Mean PR-AUC  :", round(reduced_ts_fold_results["pr_auc"].mean(), 4))
print("Std  PR-AUC  :", round(reduced_ts_fold_results["pr_auc"].std(), 4))

print("\nReduced TS OOF overall:")
print("OOF ROC-AUC :", round(roc_auc_score(y_primary_reduced_ts_final, oof_pred_reduced_ts), 4))
print("OOF PR-AUC  :", round(average_precision_score(y_primary_reduced_ts_final, oof_pred_reduced_ts), 4))

Reduced TS scale_pos_weight: 17.5714

Reduced TS fold results:


,fold,best_iteration,roc_auc,pr_auc
0,1,4,1.000000,1.000000
1,2,215,1.000000,1.000000
2,3,25,1.000000,1.000000
3,4,3,0.999419,0.982143
4,5,7,0.997133,0.935320



Reduced TS CV summary:
Mean ROC-AUC : 0.9993
Std  ROC-AUC : 0.0012
Mean PR-AUC  : 0.9835
Std  PR-AUC  : 0.028

Reduced TS OOF overall:
OOF ROC-AUC : 0.994
OOF PR-AUC  : 0.888


In [20]:
# =========================================================
# SECTION 17: TRAIN REDUCED TS MODEL ON FULL PRIMARY DATA
# AND EVALUATE ON SECONDARY DATA
# =========================================================

# Train final reduced TS model on full primary reduced TS dataset
reduced_ts_final_model = lgb.LGBMClassifier(**reduced_ts_params)
reduced_ts_final_model.fit(X_primary_reduced_ts_final, y_primary_reduced_ts_final)

# Predict on secondary reduced TS dataset
secondary_pred_reduced_ts = reduced_ts_final_model.predict_proba(X_secondary_reduced_ts_final)[:, 1]

# Ranking metrics on secondary
secondary_reduced_ts_roc_auc = roc_auc_score(y_secondary_reduced_ts_final, secondary_pred_reduced_ts)
secondary_reduced_ts_pr_auc = average_precision_score(y_secondary_reduced_ts_final, secondary_pred_reduced_ts)

print("Reduced TS secondary ranking performance:")
print("Secondary ROC-AUC :", round(secondary_reduced_ts_roc_auc, 4))
print("Secondary PR-AUC  :", round(secondary_reduced_ts_pr_auc, 4))

# Threshold scan on secondary for reduced TS model
secondary_thresholds = np.round(np.arange(0.001, 0.201, 0.001), 3)
reduced_ts_secondary_rows = []

for thr in secondary_thresholds:
    pred_label = (secondary_pred_reduced_ts >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_secondary_reduced_ts_final, pred_label).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    alert_rate = pred_label.mean()

    reduced_ts_secondary_rows.append({
        "threshold": thr,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "alert_rate": alert_rate
    })

reduced_ts_secondary_threshold_df = pd.DataFrame(reduced_ts_secondary_rows)

print("\nReduced TS best secondary thresholds by F1:")
display(reduced_ts_secondary_threshold_df.sort_values("f1", ascending=False).head(15))

print("\nReduced TS best secondary thresholds by recall (precision >= 0.20):")
display(
    reduced_ts_secondary_threshold_df.loc[reduced_ts_secondary_threshold_df["precision"] >= 0.20]
    .sort_values(["recall", "precision"], ascending=False)
    .head(15)
)

print("\nReduced TS best secondary thresholds by precision (recall >= 0.20):")
display(
    reduced_ts_secondary_threshold_df.loc[reduced_ts_secondary_threshold_df["recall"] >= 0.20]
    .sort_values(["precision", "recall"], ascending=False)
    .head(15)
)

Reduced TS secondary ranking performance:
Secondary ROC-AUC : 0.865
Secondary PR-AUC  : 0.1752

Reduced TS best secondary thresholds by F1:


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
5,0.006,167,799,2276,8,0.172878,0.954286,0.740163,0.292726,0.297231
7,0.008,150,702,2373,25,0.176056,0.857143,0.771707,0.292113,0.262154
6,0.007,157,748,2327,18,0.173481,0.897143,0.756748,0.290741,0.278462
10,0.011,124,555,2520,51,0.182622,0.708571,0.819512,0.290398,0.208923
8,0.009,141,657,2418,34,0.176692,0.805714,0.786341,0.289825,0.245538
12,0.013,112,492,2583,63,0.185430,0.640000,0.840000,0.287548,0.185846
11,0.012,117,522,2553,58,0.183099,0.668571,0.830244,0.287469,0.196615
9,0.010,131,606,2469,44,0.177748,0.748571,0.802927,0.287281,0.226769
4,0.005,173,859,2216,2,0.167636,0.988571,0.720650,0.286661,0.317538
13,0.014,103,458,2617,72,0.183601,0.588571,0.851057,0.279891,0.172615



Reduced TS best secondary thresholds by recall (precision >= 0.20):


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
197,0.198,5,20,3055,170,0.2,0.028571,0.993496,0.05,0.007692
198,0.199,5,20,3055,170,0.2,0.028571,0.993496,0.05,0.007692
199,0.200,5,20,3055,170,0.2,0.028571,0.993496,0.05,0.007692



Reduced TS best secondary thresholds by precision (recall >= 0.20):


,threshold,tp,fp,tn,fn,precision,recall,specificity,f1,alert_rate
12,0.013,112,492,2583,63,0.185430,0.640000,0.840000,0.287548,0.185846
13,0.014,103,458,2617,72,0.183601,0.588571,0.851057,0.279891,0.172615
11,0.012,117,522,2553,58,0.183099,0.668571,0.830244,0.287469,0.196615
15,0.016,89,398,2677,86,0.182752,0.508571,0.870569,0.268882,0.149846
10,0.011,124,555,2520,51,0.182622,0.708571,0.819512,0.290398,0.208923
18,0.019,74,333,2742,101,0.181818,0.422857,0.891707,0.254296,0.125231
19,0.020,70,316,2759,105,0.181347,0.400000,0.897236,0.249554,0.118769
16,0.017,84,380,2695,91,0.181034,0.480000,0.876423,0.262911,0.142769
14,0.015,95,432,2643,80,0.180266,0.542857,0.859512,0.270655,0.162154
20,0.021,66,301,2774,109,0.179837,0.377143,0.902114,0.243542,0.112923


## Final Summary

This revised notebook compared three cross-dataset modeling strategies for predicting whether a run-day will enter Growth Stage 5 within the next 7 days (`y_enter5_7`):

1. Baseline LightGBM using shared base variables only  
2. Full TS-engineered LightGBM using shared base variables plus lags, rolling means, and first differences  
3. Reduced TS-engineered LightGBM using a smaller temporal feature set built from shared base variables, `lag1`, `lag7`, and `diff1`

To make the comparison fair across datasets, I restricted all models to the feature space available in both the primary and secondary datasets. I kept `LAID` as the canonical leaf-area variable and removed the redundant `LAI` family from the modeling pipeline. I also excluded leakage-prone variables such as `GSTD`, `DAP`, treatment identifiers, and future-looking fields, and used forward fill only within each `RUNNO`.

On the primary dataset, both TS-based models outperformed the no-TS baseline under honest GroupKFold evaluation by `RUNNO`. The full TS model achieved the strongest internal performance, while the reduced TS model remained strong with a simpler and more compact feature set.

On the secondary robustness dataset, both TS-based models also outperformed the no-TS baseline. The reduced TS model achieved the strongest secondary ranking performance, with better ROC-AUC and PR-AUC than both the baseline model and the full TS model. This suggests that the reduced TS feature set transferred more effectively across datasets and was less sensitive to dataset-specific patterns than the larger full TS feature set.

Additional calibration experiments did not improve cross-dataset transfer. Sigmoid calibration did not materially improve threshold behavior, and isotonic calibration reduced robustness. This indicates that the main challenge is not only probability calibration, but also distribution shift between datasets.

Overall, the reduced TS-engineered LightGBM is the strongest final model from this notebook. It provides the best balance between strong primary performance, improved transfer to the secondary dataset, and a simpler, more stable feature set. These results suggest that the model is most useful as a risk-ranking and alert prototype, while also showing that deployment thresholds would still need recalibration on representative company data before operational use.